In [6]:
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path

try:
    from mnist_realtime_utils import prepare_digit_for_cnn
except ModuleNotFoundError:
    from ml.mnist_realtime_utils import prepare_digit_for_cnn

In [7]:
MODEL_PATH = Path("CNN_MNIST_realtime.keras")

In [8]:
def build_cnn_mnist_model():
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input((28, 28, 1)))
    model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'))
    model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu'))
    model.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(64, activation='relu'))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dropout(0.2))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy'],
    )
    return model

In [9]:
if MODEL_PATH.exists():
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f"Loaded saved model: {MODEL_PATH}")
else:
    (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
    train_images = train_images.reshape((60000, 28, 28, 1)).astype("float32") / 255.0
    test_images = test_images.reshape((10000, 28, 28, 1)).astype("float32") / 255.0

    model = build_cnn_mnist_model()
    model.fit(train_images, train_labels, epochs=10)
    model.evaluate(test_images, test_labels)
    model.save(MODEL_PATH)
    print(f"Saved model: {MODEL_PATH}")

Loaded saved model: CNN_MNIST_realtime.keras


In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION) # use MacBook FaceTime HD camera
if not cap.isOpened():
    print("Cannot open webcam.")

last_prediction = "Press C to capture"

while True:
    (ret,frame) = cap.read()
    if not ret:
        print("Frame read error")
        break

    flip_frame = cv2.flip(frame,1) # mirror preview
    (height, width, _) = flip_frame.shape
    (center_x, center_y) = (width//2, height//2)
    roi = flip_frame[center_y - 150:center_y + 150, center_x - 150:center_x + 150]
    cv2.rectangle(flip_frame,(center_x-150,center_y-150),(center_x + 150, center_y+ 150),(0,0,255),2)
    cv2.putText(flip_frame, last_prediction, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Webcam", flip_frame)

    key = cv2.waitKey(1) & 0xFF

    if key in (ord('c'), ord('C')):
        gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray_image = np.flip(gray_image, axis=1)
        cv2.imwrite('gray_image.png', gray_image)
        gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
        (_, otsh_thresh) = cv2.threshold(gaussian_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cv2.imshow("OTSU",otsh_thresh)
        kernel = np.ones((5,5), np.uint8)
        erosion = cv2.erode(otsh_thresh, kernel, iterations=5)
        cv2.imshow("EROSION", erosion)
        cv2.imwrite("digit_binary_image.png",erosion)

        img = erosion
        (h, w) = img.shape[:2]
        digit_mask = np.uint8(img < 128) * 255
        contours, _ = cv2.findContours(digit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            contour = max(contours, key=cv2.contourArea)
            x, y, crop_w, crop_h = cv2.boundingRect(contour)
            padding = 10
            x1 = max(0, x - padding)
            y1 = max(0, y - padding)
            x2 = min(w, x + crop_w + padding)
            y2 = min(h, y + crop_h + padding)
            crop = img[y1:y2, x1:x2]
        else:
            crop = img
        cv2.imshow("CROP", crop)
        reversed_image = cv2.bitwise_not(crop)
        cv2.imshow("REVERSED_IMAGE", reversed_image)
        cv2.imwrite("IMAGE_FOR_TEST.png", reversed_image)

        input_image = prepare_digit_for_cnn(reversed_image)
        probabilities = model.predict(input_image, verbose=0)[0]
        predicted_digit = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_digit])
        last_prediction = f"Prediction: {predicted_digit} ({confidence:.2%})"
        print(last_prediction)


    if cv2.waitKey(30) == 27:
        break
cap.release()
cv2.destroyAllWindows()

: 